In [ ]:
import pandas as pd
import json
import sys

# NASA Data Research

In [ ]:
# Creating a DataFrame for all NEOs
filtered_data = pd.DataFrame(data['near_earth_objects'])
filtered_data.drop('links', axis=1, inplace=True) # Dropping links for security reasons



# Converting estimated diameter to meters and getting it from json
all_neos_df = filtered_data.copy()
all_neos_df['estimated_diameter_meters_max'] = all_neos_df.estimated_diameter.apply(lambda x: x['kilometers']['estimated_diameter_max'] * 1000)
all_neos_df['estimated_diameter_meters_min'] = all_neos_df.estimated_diameter.apply(lambda x: x['kilometers']['estimated_diameter_min'] * 1000)
all_neos_df.drop('estimated_diameter', axis = 1, inplace=True) # dropping json as we got all we needed

# Creating orbital data as a separate DataFrame. We will not parse it since we don't know if we will need it
orbital_data = all_neos_df[['id', 'neo_reference_id', 'name', 'name_limited', 'orbital_data']].copy()

# Creating close approach data as a separate DataFrame
close_approach_data = pd.DataFrame()

for i in range(len(all_neos_df)):
    temp_df = pd.DataFrame(all_neos_df.close_approach_data[i])
    temp_df['neo_reference_id'] = all_neos_df.neo_reference_id[i]
    temp_df['id'] = all_neos_df.id[i]
    temp_df['name'] = all_neos_df.name[i]
    temp_df['is_potentially_hazardous_asteroid'] = all_neos_df.is_potentially_hazardous_asteroid[i]
    try:
        temp_df['relative_velocity_kph'] = temp_df.relative_velocity.apply(lambda x: x['kilometers_per_hour'])
        temp_df['miss_distance_meters'] = temp_df.miss_distance.apply(lambda x: x['kilometers'] * 1000)
        close_approach_data = pd.concat([close_approach_data, temp_df])
    except:
        pass

# At the end we will get full data as all_neos_df